In [11]:
import pandas as pd 
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# Login using e.g. `huggingface-cli login` to access this dataset
df = pd.read_csv("hf://datasets/ZakyF/PRDECT-ID/PRDECT-ID Dataset.csv")

In [13]:
df.head()

,Category,Product Name,Location,Price,Overall Rating,Number Sold,Total Review,Customer Rating,Customer Review,Sentiment,Emotion
0,Computers and Laptops,Wireless Keyboard i8 Mini TouchPad Mouse 2.4G ...,Jakarta Utara,53500,4.9,5449,2369,5,Alhamdulillah berfungsi dengan baik. Packaging...,Positive,Happy
1,Computers and Laptops,PAKET LISENSI WINDOWS 10 PRO DAN OFFICE 2019 O...,Kota Tangerang Selatan,72000,4.9,2359,1044,5,"barang bagus dan respon cepat, harga bersaing ...",Positive,Happy
2,Computers and Laptops,SSD Midasforce 128 Gb - Tanpa Caddy,Jakarta Barat,213000,5.0,12300,3573,5,"barang bagus, berfungsi dengan baik, seler ram...",Positive,Happy
3,Computers and Laptops,ADAPTOR CHARGER MONITOR LCD LED TV LG merek LG...,Jakarta Timur,55000,4.7,2030,672,5,bagus sesuai harapan penjual nya juga ramah. t...,Positive,Happy
4,Computers and Laptops,ADAPTOR CHARGER MONITOR LCD LED TV LG merek LG...,Jakarta Timur,55000,4.7,2030,672,5,"Barang Bagus, pengemasan Aman, dapat Berfungsi...",Positive,Happy


In [18]:
models_to_test = [
    "Qwen/Qwen2.5-0.5B-Instruct",  # <-- Ubah ke versi 2.5 yang benar
    "google/gemma-1.1-2b-it",
    "microsoft/phi-3-mini-4k-instruct"
]

In [19]:
def get_soft_prob_vector(model, tokenizer, review_text):
    prompt = f"""Klasifikasikan ulasan produk elektronik ini ke dalam salah satu kategori:
A: Positif
B: Netral
C: Negatif
D: Khusus/Sarkas

Ulasan: "{review_text}"
Jawaban (Pilih A/B/C/D):"""

    inputs = tokenizer(prompt, return_tensors="pt")
    
    # Deteksi perangkat (Aman untuk CPU ARM64 di Ubuntu VM)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)

    # Ambil logits token terakhir
    next_token_logits = outputs.logits[0, -1, :]

    # Ambil ID untuk token A, B, C, D
    # Menggunakan trik aman untuk berbagai jenis tokenizer
    token_A = tokenizer.encode("A", add_special_tokens=False)[-1]
    token_B = tokenizer.encode("B", add_special_tokens=False)[-1]
    token_C = tokenizer.encode("C", add_special_tokens=False)[-1]
    token_D = tokenizer.encode("D", add_special_tokens=False)[-1]

    selected_logits = torch.tensor([
        next_token_logits[token_A],
        next_token_logits[token_B],
        next_token_logits[token_C],
        next_token_logits[token_D]
    ])

    # Softmax untuk mendapatkan probabilitas 4 Dimensi
    soft_probs = torch.nn.functional.softmax(selected_logits, dim=0)
    return [round(p, 3) for p in soft_probs.tolist()]

In [20]:
# 3. Looping Eksekusi dan Perbandingan
results = []

for model_name in models_to_test:
    print(f"\n[{model_name}] Loading model...")
    # Load model ke memori (Otomatis deteksi CPU)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    for index, row in df.iterrows():
        review = row["Customer Review"]
        vector = get_soft_prob_vector(model, tokenizer, review)
        
        results.append({
            "Model": model_name.split("/")[-1],
            "Ulasan": review[:30] + "...", # Potong teks agar rapi saat diprint
            "Vektor [S+, S~, S-, SK]": vector
        })

    # 4. Bersihkan memori VM sebelum meload model berikutnya
    print(f"[{model_name}] Selesai. Membersihkan memori...")
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None


[Qwen/Qwen2.5-0.5B-Instruct] Loading model...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 